# From an Odin retrieval to its fitted spectra

Also save the matching L2I selection as `diagnostics-selection.json`. Both selections must explicitly match release, project, frequency mode and inversion mode. This template handles scan-row and STW-row diagnostic layouts without flattening channel arrays.

Download `selection.json` from Odin Data Access and save it next to this notebook. A planned release has no selectable files yet; do not substitute a migration directory or invented object URL. Read the selected release's schema, license and citation first.

Install the tested environment in a terminal: `python -m pip install duckdb==1.4.4 matplotlib==3.10.8 numpy==2.4.2 jupyterlab`. Restart the kernel after changing packages. Python 3.12 is tested.

The next cell downloads the **complete selected files**, up to a 64 MiB tutorial cap, verifies SHA256, and reuses a verified local cache. The website's date selection finds overlapping files; the files may contain observations outside those dates. No AWS account, keys or cloud query service is used. Exact bytes and rows depend on your selection; synthetic validation is described in the maintainer guide.

Keep `selection.json` and the generated provenance JSON with your result.


In [ ]:
import hashlib
import importlib.metadata
import json
import re
import sys
from pathlib import Path
from urllib.parse import urlsplit
from urllib.request import HTTPRedirectHandler, build_opener

import duckdb
import matplotlib.pyplot as plt
import numpy as np

# Tutorial safety limit, not an estimate of the whole dataset or query cost.
MAX_DOWNLOAD_BYTES = 64 * 1024 * 1024
CACHE = Path("odin-parquet-cache")
CACHE.mkdir(exist_ok=True)
TEMPLATE_VERSION = "odin-parquet-notebooks-1"
SELECTION_SOURCES = {}


class HTTPSRedirects(HTTPRedirectHandler):
    def redirect_request(self, request, fp, code, message, headers, new_url):
        target = urlsplit(new_url)
        if target.scheme != "https" or target.username or target.password:
            raise ValueError("Refusing a redirect outside anonymous HTTPS.")
        return super().redirect_request(request, fp, code, message, headers, new_url)


https_opener = build_opener(HTTPSRedirects())


def read_selection(filename):
    raw = Path(filename).read_bytes()
    selection = json.loads(raw)
    if selection.get("schema_version") != 1 or not selection.get("files"):
        raise ValueError("Use a non-empty version 1 selection exported by Odin Data Access.")
    for name in ("release", "dataset"):
        if not isinstance(selection.get(name), dict) or not selection[name].get("id"):
            raise ValueError(f"Missing {name} identity.")
    if not selection["release"].get("license") or not selection["release"].get("citation"):
        raise ValueError("The selection must retain its license and citation.")
    seen = set()
    for entry in selection["files"]:
        url = entry["url"]
        parsed = urlsplit(url)
        if (parsed.scheme != "https" or not parsed.hostname or parsed.username
                or parsed.password or parsed.query or parsed.fragment):
            raise ValueError("Expected permanent anonymous HTTPS object URLs.")
        if url in seen:
            raise ValueError("Duplicate object in selection.")
        seen.add(url)
        if not re.fullmatch(r"[0-9a-f]{64}", entry["sha256"]):
            raise ValueError("Missing lowercase SHA256 checksum.")
        for key in ("size_bytes", "row_count"):
            if type(entry[key]) is not int or entry[key] < 0:
                raise ValueError(f"Invalid {key}.")
    if sum(f["size_bytes"] for f in selection["files"]) > MAX_DOWNLOAD_BYTES:
        raise ValueError("Selection exceeds the tutorial cap; choose fewer files.")
    print("Release:", selection["release"]["id"], "Dataset:", selection["dataset"]["id"])
    print("Processing identity:", {k: selection["dataset"].get(k) for k in
          ("project", "product", "frequency_mode", "inversion_mode")})
    print("Declared files/bytes:", len(selection["files"]),
          sum(f["size_bytes"] for f in selection["files"]))
    print("Selection SHA256:", hashlib.sha256(raw).hexdigest())
    SELECTION_SOURCES[id(selection)] = {"filename": filename, "sha256": hashlib.sha256(raw).hexdigest()}
    return selection


def verified_files(selection):
    paths = []
    for entry in selection["files"]:
        destination = CACHE / (entry["sha256"] + ".parquet")
        if not destination.exists():
            temporary = destination.with_suffix(".part")
            try:
                count = 0
                with https_opener.open(entry["url"], timeout=60) as response, temporary.open("wb") as target:
                    if urlsplit(response.url).scheme != "https":
                        raise ValueError("HTTPS is required after redirects.")
                    while block := response.read(1024 * 1024):
                        count += len(block)
                        if count > entry["size_bytes"] or count > MAX_DOWNLOAD_BYTES:
                            raise ValueError("Download exceeded the declared byte count.")
                        target.write(block)
                with temporary.open("rb") as source:
                    digest = hashlib.file_digest(source, "sha256").hexdigest()
                if temporary.stat().st_size != entry["size_bytes"] or digest != entry["sha256"]:
                    raise ValueError("Downloaded bytes do not match the release checksum/size.")
                temporary.replace(destination)
            finally:
                temporary.unlink(missing_ok=True)
        with destination.open("rb") as source:
            digest = hashlib.file_digest(source, "sha256").hexdigest()
        if destination.stat().st_size != entry["size_bytes"] or digest != entry["sha256"]:
            raise ValueError("Cached file failed verification; inspect or remove that cached file.")
        paths.append(str(destination))
    return paths


def open_table(selection, table_name):
    paths = verified_files(selection)
    # Partition values remain in selection.json. No directory-based type inference.
    relation = con.read_parquet(paths, hive_partitioning=False, union_by_name=False)
    relation.create_view(table_name, replace=True)
    expected_rows = sum(f["row_count"] for f in selection["files"])
    actual_rows = con.execute(f'SELECT count(*) FROM "{table_name}"').fetchone()[0]
    if actual_rows != expected_rows:
        raise ValueError("File row counts differ from the selected manifest.")
    return dict(con.execute(f'DESCRIBE "{table_name}"').fetchall()[i][:2]
                for i in range(len(relation.columns)))


def column_unit(selection, name):
    matches = [c for c in selection["dataset"]["columns"] if c["name"] == name]
    if len(matches) != 1 or not matches[0].get("unit"):
        raise ValueError(f"Read the release dictionary and supply a verified unit for {name}.")
    return matches[0]["unit"]


def require_columns(schema, *names):
    missing = set(names) - schema.keys()
    if missing:
        raise ValueError(f"This notebook does not support this physical schema; missing {sorted(missing)}.")


def provenance(selection, filename, analysis):
    receipt = {"template_version": TEMPLATE_VERSION, "analysis": analysis,
               "selection_source": SELECTION_SOURCES[id(selection)],
               "download_cap_bytes": MAX_DOWNLOAD_BYTES,
               "selection": selection, "python": sys.version,
               "packages": {p: importlib.metadata.version(p)
                            for p in ("duckdb", "matplotlib", "numpy")}}
    Path(filename).write_text(json.dumps(receipt, indent=2) + "\n", encoding="utf-8")


con = duckdb.connect()
con.execute("SET TimeZone = 'UTC'")
selection = read_selection("selection.json")
schema = open_table(selection, "observations")
print("Physical columns:", schema)


## Run the example

Adjust the documented parameters below. The notebook stops for unsupported or ambiguous schemas instead of changing scientific identities or guessing units.

In [ ]:
if selection["dataset"]["level"] != "L2":
    raise ValueError("selection.json must describe L2; save the matching L2I selection separately.")
diag_selection = read_selection("diagnostics-selection.json")
if diag_selection["dataset"]["level"] != "L2I":
    raise ValueError("diagnostics-selection.json must describe L2I.")
if (sum(f["size_bytes"] for f in selection["files"])
        + sum(f["size_bytes"] for f in diag_selection["files"])) > MAX_DOWNLOAD_BYTES:
    raise ValueError("Combined selections exceed the tutorial cap.")
if selection["release"]["id"] != diag_selection["release"]["id"]:
    raise ValueError("Use matching L2 and L2I releases; do not infer cross-release equivalence.")
for key in ("project", "frequency_mode", "inversion_mode"):
    expected = selection["dataset"].get(key)
    if expected is None or expected != diag_selection["dataset"].get(key):
        raise ValueError(f"Both selections must explicitly agree on {key}.")
diag_schema = open_table(diag_selection, "diagnostics")
require_columns(schema, "scan_id")
require_columns(diag_schema, "scan_id", "stw", "fit_spectrum")
SCAN_ID = con.execute('SELECT min(scan_id) FROM observations').fetchone()[0]
fields = ["stw", "fit_spectrum"]
if "processing_error" in diag_schema:
    fields.append("processing_error")
records = con.execute('SELECT ' + ', '.join(fields)
                      + ' FROM diagnostics WHERE scan_id = ?', [SCAN_ID]).fetchall()
print("L2 scan:", SCAN_ID, "Matching L2I rows:", len(records))
diagnostic_layout = "unavailable"
plotted_stw = None
if not records:
    print("No matching diagnostics in the selected files.")
elif any(len(r) > 2 and r[2] is True for r in records):
    print("Processing failure is recorded. Do not interpret null diagnostics as zeros or success.")
elif all(r[0] is None or r[1] is None for r in records):
    print("Diagnostics are missing; no fitted spectrum is plotted.")
else:
    if any(r[0] is None or r[1] is None for r in records):
        raise ValueError("Mixed present/missing diagnostic rows; inspect the release-specific failure contract.")
    if isinstance(records[0][0], list):
        if len(records) != 1:
            raise ValueError("Multiple scan-row diagnostics are ambiguous; resolve duplicate/version identity.")
        diagnostic_layout = "scan-row"
        stws, spectra = records[0][:2]
    else:
        diagnostic_layout = "stw-row"
        stws, spectra = [r[0] for r in records], [r[1] for r in records]
    if any(type(stw) is not int for stw in stws):
        raise ValueError("Every STW must be a present integer identifier.")
    if len(stws) != len(spectra) or len(set(stws)) != len(stws):
        raise ValueError("STW/spectrum alignment or uniqueness failed.")
    if not spectra or not all(isinstance(s, list) for s in spectra):
        raise ValueError("Unsupported fitted-spectrum nesting; inspect the release dictionary.")
    # Preserve stored channel order. This plot is not a reconstructed frequency axis.
    chosen_index = min(range(len(stws)), key=lambda index: stws[index])
    plotted_stw = stws[chosen_index]
    fitted = np.asarray(spectra[chosen_index], dtype=float)
    if fitted.ndim != 1:
        raise ValueError("Expected one vector of channels for this STW.")
    spectrum_unit = column_unit(diag_selection, "fit_spectrum")
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.plot(np.arange(len(fitted)), fitted)
    ax.set(xlabel="Stored channel index (not frequency)",
           ylabel=f"Fitted spectrum [{spectrum_unit}]",
           title=f"Scan {SCAN_ID}, STW {plotted_stw}")
    fig.tight_layout()
    fig.savefig("odin-fitted-spectrum.png", dpi=150)
    plt.show()
    print("STWs / chosen spectrum channels:", len(stws), len(fitted))
analysis = {"notebook": "diagnostics", "scan_id": SCAN_ID, "layout": diagnostic_layout,
            "plotted_stw": plotted_stw, "stw_selection": "minimum integer STW",
            "join_keys": ["release", "project", "frequency_mode", "inversion_mode", "scan_id"],
            "l2_selection": SELECTION_SOURCES[id(selection)],
            "l2i_selection": SELECTION_SOURCES[id(diag_selection)],
            "channel_axis": "stored index", "quality_filter": None}
provenance(selection, "diagnostics-l2-provenance.json", analysis)
provenance(diag_selection, "diagnostics-l2i-provenance.json", analysis)


## Interpretation

These examples inspect stored data; they do not establish scientific validity or mission-wide completeness. Null values remain missing. Follow the release's product-specific quality and uncertainty guidance before drawing scientific conclusions. Preserve original coordinate/array order when using averaging kernels or other matrices; sorting a plotted profile does not reorder its kernel axes. Cite the release and processing project recorded in your selection. See [the analysis guide](https://odin-smr.org/data_access/parquet-guide) for formats and further analysis.